# Lesson 4.2 — Eval Datasets and Mixed-Method Test Frameworks

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Build a labeled eval dataset with edge cases
- Combine exact-match, rubric, and LLM-as-judge scoring (mixed methods)
- Run the dataset through a scorer and report per-case results

**You will produce:** A JSONL eval dataset and a mixed-method scorer producing a scorecard.

## Concepts

An **eval dataset** is a curated set of inputs with known-good expectations (a 'gold set'). Good sets include **edge cases** and failure-prone inputs, not just happy paths, and are versioned.

**Mixed-method testing** combines scoring styles because no single one fits every task:
- **Exact/programmatic** — deterministic checks (label match, JSON schema, regex, citation present). Fast and cheap; use whenever the answer is structured.
- **Rubric** — human- or code-applied criteria for open-ended outputs.
- **LLM-as-judge** — a model grades outputs against a rubric; scalable for subjective quality but must itself be validated against human labels.

The architect designs the dataset and chooses which method scores which field, so the harness is both trustworthy and affordable.

> **Reference:** [Anthropic — Create strong empirical evaluations](https://docs.anthropic.com/en/docs/test-and-evaluate/develop-tests)

In [ ]:
# !pip install anthropic
import os, json, time, random

try:
    import anthropic
    if os.environ.get("ANTHROPIC_API_KEY"):
        client = anthropic.Anthropic(); LIVE = True
    else:
        client = None; LIVE = False
except Exception as e:
    client = None; LIVE = False
    print("anthropic SDK unavailable:", e)

MODELS = {"opus":"claude-opus-4-20250514","sonnet":"claude-sonnet-4-20250514","haiku":"claude-3-5-haiku-20241022"}

def ask_claude(prompt, model=MODELS["sonnet"], system=None, max_tokens=512):
    if not LIVE:
        # Deterministic mock: echo a label when the prompt asks for classification
        return f"[MOCK {model}] {prompt[:60]}"
    msg = client.messages.create(model=model, max_tokens=max_tokens,
        system=system or "You are a helpful assistant.",
        messages=[{"role":"user","content":prompt}])
    return msg.content[0].text

print("LIVE" if LIVE else "MOCK", "mode ready")

## Code-Along: Step 1 — Create and persist a labeled eval dataset

We write a small JSONL dataset including an ambiguous edge case. JSONL (one JSON object per line) is the standard eval format and is easy to version and append to.

In [ ]:
dataset = [
  {"id":"c1","input":"MRI after 8 weeks failed PT","gold":"approve"},
  {"id":"c2","input":"MRI, no PT attempted","gold":"deny"},
  {"id":"c3","input":"MRI, PT status not documented","gold":"needs_info"},  # edge case
  {"id":"c4","input":"Emergency head CT after trauma","gold":"approve"},
]
with open("eval_dataset.jsonl","w",encoding="utf-8") as fh:
    for row in dataset: fh.write(json.dumps(row)+"\n")
print(f"wrote {len(dataset)} cases to eval_dataset.jsonl")

## Code-Along: Step 2 — Score with mixed methods

We run each case through the system (mock or live), apply exact-match on the label, and demonstrate an LLM-as-judge rubric check (also mock-safe). The scorecard aggregates both.

In [ ]:
LABELS = {"approve","deny","needs_info"}
def predict(case_input):
    out = ask_claude(f"Classify coverage as approve/deny/needs_info: {case_input}. One word.",
                     model=MODELS['haiku'], max_tokens=10)
    # extract a label if present; else fall back (mock mode won't contain one)
    for lab in LABELS:
        if lab in out.lower(): return lab
    return "needs_info"  # safe default

def exact_match(pred, gold): return pred == gold

def judge_rubric(case_input, pred):
    # LLM-as-judge (mock-safe). Production: validate judge vs human labels first.
    verdict = ask_claude(f"Is '{pred}' a reasonable triage for '{case_input}'? yes/no.",
                         model=MODELS['haiku'], max_tokens=5)
    return "no" not in verdict.lower()

scorecard = []
for row in dataset:
    pred = predict(row['input'])
    scorecard.append({"id":row['id'],"pred":pred,"gold":row['gold'],
                      "exact":exact_match(pred,row['gold']),
                      "judge_ok":judge_rubric(row['input'],pred)})
for s in scorecard: print(s)

## Exercise

Expand the dataset to 10 cases with at least 3 edge cases (ambiguous, adversarial, out-of-scope). Add a **schema check** scorer (prediction must be one of the valid labels) and produce a summary row: exact-match accuracy, judge-agreement rate, and schema-valid rate. Note where exact-match and the judge disagree — those are your most informative cases.

In [ ]:
# TODO: extend dataset to >=10 rows with edge cases
# TODO: add schema_valid(pred) scorer
# TODO: print summary: exact accuracy, judge agreement, schema valid rate; list disagreements

## Key Takeaways

- A gold set with edge cases is the backbone of trustworthy evaluation.
- Mix scoring methods: exact for structure, rubric/LLM-judge for open-ended quality.
- Validate LLM-as-judge against human labels before trusting it.
- JSONL datasets are versionable and append-friendly — treat them like code.

---

**Next:** [Lesson 4.3 — A/B Testing and Iterative Improvement](./lesson-4.3-ab-testing-iteration.ipynb)